### Semana 11: arquitectura multiagente, contratos y evaluación

Curso CC-0F4. Sesión teórica: dos horas.

Pregunta de la sesión: ¿qué aporta dividir una tarea entre agentes si la recuperación inicial es la misma, y qué nuevas fallas pueden aparecer?

Este cuaderno diferencia la recuperación de HotpotQA, las pruebas de contratos y la inferencia real con Qwen. No presenta la orquestación determinista como un agente LLM.

#### 1. Preparar el entorno

Los experimentos usan los archivos originales de HotpotQA preparados por el docente con `python -m Semana11.multiagente_real --preparar`. La carga debe fallar si falta el archivo o no coincide su hash. No se fabrican ejemplos para ejecutar el cuaderno.

In [ ]:
from pathlib import Path
import json, os, sys
root = Path.cwd().resolve()
for candidate in (root, *root.parents):
    if (candidate / 'Semana11/multiagente_real.py').is_file():
        root = candidate
        break
else:
    raise FileNotFoundError('Inicia JupyterLab desde el repositorio CC-0F4')
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from Semana11.multiagente_real import (check_dataset, ranked_evidence, retrieval_rows,
    summarize_retrieval, support_recall, gold_facts, evaluate_retrieval,
    validate_answer, select_pilot, pilot, MODEL_ID)
from Semana11.contratos import Blackboard, ContractError, make_message, validate_selection
examples = check_dataset()
print('Directorio:', root)
print('Preguntas originales verificadas:', len(examples))

#### 2. HotpotQA: pregunta, contexto y evidencia original

HotpotQA registra pregunta, respuesta humana, párrafos distractores y hechos de apoyo. Cada hecho de apoyo se identifica por el título del documento y el índice de la oración. La variante distractor no representa recuperación abierta sobre Wikipedia.

El recuperador utilizará solamente `question` y `context`. No debe acceder a `answer` ni a `supporting_facts` antes de la evaluación.

In [ ]:
# Inspección de un registro humano real, sin usar la respuesta como entrada al recuperador.
example = examples[0]
print('ID:', example['_id'])
print('Tipo:', example['type'], '| dificultad:', example['level'])
print('Pregunta:', example['question'])
print('Número de párrafos:', len(example['context']))
print('Número de hechos gold:', len(example['supporting_facts']))

#### 3. Agente único y supervisor con trabajadores

Agente único: pregunta y evidencia disponible, una generación de respuesta y citas.

Supervisor: primera generación para seleccionar identificadores, validación y registro, segunda generación para responder con fragmentos seleccionados.

El supervisor actual aplica reglas en Python. Los trabajadores solo se consideran dirigidos por LLM cuando se ejecuta el modo Qwen. El controlador no es por sí solo un modelo autónomo.

#### 4. Acciones y mensajes observables

Una transición de coordinación puede modelarse como una función de estado, entrada y mensaje:

\[ s_{t+1} = F(s_t, m_t, o_t). \]

El mensaje aceptado debe indicar `question_id`, `role`, `selected_ids` y huellas de procedencia. La pizarra permite reconstruir el orden de aceptación. Un identificador no observado debe ser rechazado, aunque aparezca en una respuesta JSON correctamente formada.

In [ ]:
observed = ranked_evidence(example, top_k=14)
print('Oraciones disponibles:', len(observed))
for row in observed[:4]:
    print(row['id'], row['title'], row['sent_id'], row['text'][:140])
assert all(isinstance(row['id'], str) for row in observed)

#### 5. Contrato de traspaso y pizarra

La pizarra guarda mensajes aceptados y no permite cambiar los datos validados a posteriori. Se añade una huella SHA-256 del texto recuperado y su procedencia. Esta verificación de integridad no acredita relevancia semántica.

In [ ]:
board = Blackboard(example['_id'])
selected = [row['id'] for row in observed[:2]]
msg = make_message(example['_id'], 'evidence_worker', selected, observed)
board.append(msg, observed)
print('Mensajes aceptados:', len(board.messages))
print('IDs registrados:', board.messages[0]['selected_ids'])
assert board.messages[0]['selected_ids'] == selected

#### 6. Validación: el supervisor debe poder rechazar

Una regla de seguridad debe estar en código, no depender solo de que el modelo siga instrucciones. Compara una cita observada y una no observada. La siguiente celda inyecta errores para probar contratos, no añade preguntas sintéticas al benchmark.

In [ ]:
# Inyección de falla de contrato, separada de cualquier resultado experimental.
try:
    validate_selection(['ID_NO_OBSERVADO'], observed)
except ContractError as exc:
    print('Rechazo esperado:', exc)
else:
    raise AssertionError('El contrato aceptó una cita inexistente')
valid, reason, answer, cited = validate_answer(
    {'answer':'Revisar evidencia', 'cited_ids': [observed[0]['id']]}, observed)
print('Finalización estructural:', valid, reason)
assert valid  # No demuestra que la respuesta de ejemplo sea factual.

#### 7. Recuperación BM25 sin filtrar las anotaciones

El algoritmo recorre las oraciones de los párrafos suministrados con una pregunta. Calcula frecuencia de términos, frecuencia documental y una puntuación BM25. El gold queda fuera de la función de recuperación.

La unidad de evaluación es la oración, no el pasaje completo. Dos oraciones del mismo documento son dos unidades distintas.

In [ ]:
ranked = ranked_evidence(example, 14)
assert ranked == ranked_evidence(example, 14)
print('Ranking determinista para:', example['_id'])
for x in ranked[:3]:
    print(x['id'], f"BM25={x['bm25']:.3f}", x['text'][:120])

#### 8. Métricas para evidencia de varios saltos

Con G como conjunto de hechos de apoyo humanos y R como oraciones recuperadas:

\[ \mathrm{Recall} = |G \cap R|/|G|. \]

AnyHit vale uno si al menos una oración gold está en R. AllHit vale uno solo si todas las oraciones gold están presentes. No equipares AnyHit con capacidad de contestar la pregunta.

In [ ]:
# La anotación original se abre únicamente después del ranking.
print('Respuesta humana:', example['answer'])
print('Hechos humanos:', sorted(gold_facts(example)))
print('Recall de soporte:', round(support_recall(example, ranked), 4))
print('Observado:', [(x['title'],x['sent_id']) for x in ranked[:4]])

#### 9. Evaluación completa, sin simulación

La siguiente celda ejecuta BM25 en las 7.405 preguntas reales y escribe resultados por pregunta. Puede reutilizar el resultado generado previamente por el docente; no se sustituyen preguntas de prueba por un conjunto ilustrativo.

In [ ]:
summary = evaluate_retrieval()
print('Preguntas:', summary['questions'])
for metric, value in summary['retrieval'].items():
    print(metric, value)
assert summary['questions'] == 7405

#### 10. Desglose por tipo y dificultad

Los promedios globales pueden ocultar comportamientos distintos entre preguntas de tipo `bridge` y `comparison`, o entre niveles de dificultad. El desglose proviene de los metadatos originales, no de etiquetas construidas después de observar predicciones.

In [ ]:
for name, group in summary['by_type'].items():
    print('Tipo:', name, '| preguntas:', group['questions'],
          '| AllHit:', round(group['all_support_rate'], 4))
for name, group in summary['by_level'].items():
    print('Nivel:', name, '| preguntas:', group['questions'],
          '| Recall:', round(group['mean_support_recall'], 4))

#### 11. Limites de interpretación y costo

El modelo local genera tokens de entrada y salida y consume tiempo de inferencia. El supervisor con trabajadores necesita normalmente dos llamadas, mientras que el agente único necesita una. La comparación no iguala automáticamente presupuesto ni complejidad.

Ni EM/F1 sobre respuestas ni las coincidencias de citas bastan para establecer la veracidad de todas las afirmaciones. Se requiere evaluación de respaldo factual independiente para una conclusión de grounding.

#### 12. Piloto opcional, nunca simulado

La siguiente celda solo ejecuta Qwen cuando el docente activa `CC0F4_RUN_REAL_LLM=1`. Es necesario descargar el checkpoint en la máquina antes de la clase. Los resultados inválidos deben permanecer registrados como fallos.

In [ ]:
if os.getenv('CC0F4_RUN_REAL_LLM', '0') == '1':
    count = int(os.getenv('CC0F4_S11_LLM_N', '16'))
    print(json.dumps(pilot(count, os.getenv('CC0F4_S11_MODEL', MODEL_ID)),
                     ensure_ascii=False, indent=2))
else:
    print('Piloto LLM NO EJECUTADO. No se atribuyen métricas a Qwen.')

#### 13. Cierre de la teoría

Explica por qué una cita estructuralmente válida puede ser equivocada, qué falla cuando se conserva una referencia mutable en la pizarra y por qué AllHit es una medida más estricta que AnyHit.

Diferencia claramente los tres resultados de aprendizaje: recuperación de evidencia, cumplimiento de contratos y respuestas del modelo. Ninguna prueba unitaria debe confundirse con una evaluación científica de agentes.